# AlphaTrain Pillar 2c: Pure Preference Value Head (Colab)

Scalar value head trained with **ranking loss only** (no categorical CE).
Eliminates the loss conflict that caused overfitting in 2b.

Value head outputs a dimensionless scalar — only relative ordering matters.
For MCTS, this is all we need: which leaf is better?

Upload `alphatrain_colab_td.tar.gz` to Google Drive first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/alphatrain_colab_td.tar.gz /content/
!cd /content && tar xzf alphatrain_colab_td.tar.gz
!pip install -q numpy numba pytest scipy

In [ ]:
import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Data: {os.path.getsize("/content/alphatrain/data/alphatrain_pairwise.pt") / 1e6:.0f} MB')
!cd /content && python -m pytest alphatrain/tests/ -v --tb=short

In [ ]:
%cd /content
# Pure ranking: scalar value head, warm start from Pillar 2a
# Batch 8192: A100/H100 use ~31% VRAM at 4096, 8192 is safe 2x
# LR scaled linearly: 3e-4 * 2 = 6e-4
!python -m alphatrain.train \
    --tensor-file alphatrain/data/alphatrain_pairwise.pt \
    --gpu-data --amp --compile \
    --value-bins 1 \
    --resume alphatrain/data/alphatrain_2a_best.pt --warm-start \
    --epochs 20 --batch-size 8192 --lr 6e-4 --warmup-epochs 2 \
    --rank-weight 1.0 \
    --copy-to /content/drive/MyDrive/alphatrain_td_best.pt \
    --save-dir /content/checkpoints/alphatrain_td

In [ ]:
!cp /content/checkpoints/alphatrain_td/latest.pt /content/drive/MyDrive/alphatrain_td_latest.pt
print('Saved to Drive')

In [ ]:
import sys
sys.path.insert(0, '/content')

# Evaluate standalone policy
!cd /content && python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/alphatrain_td/best.pt \
    --games 20 --seed 42

# Diagnose value head quality (key metric: rank correlation)
!cd /content && python -m alphatrain.scripts.debug_value_head \
    --model /content/checkpoints/alphatrain_td/best.pt --seed 42